In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
import keras

# Add project root to sys.path
repo_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

from models.conv_hybrid_vit import build_conv_hybrid_vit
from models.encoder import ClassifierWithEncoder
from training.losses import l2_normalize


## 1. Load Model and Prototype Banks

In [ ]:
# Configuration based on the primary model
input_size = 384
d_model = 128
n_patches = (input_size // 32) ** 2  # 144

# 1. Build the model
encoder = build_conv_hybrid_vit(
    input_shape=(input_size, input_size, 3),
    backbone_name='efficientnetv2_b0',
    d_model=d_model,
    n_patches=n_patches,
    depth=4,
    num_heads=8,
    mlp_dim=384,
    token_attention_mode='ca',
    pool_mode='gap',
    use_sinusoidal_pe=True
)

model = ClassifierWithEncoder(encoder=encoder, num_classes=2)

# Run dummy input to build weights
dummy_input = tf.zeros((1, input_size, input_size, 3))
_ = model(dummy_input, training=False)

# Load weights
weights_path = 'classifier_y0taaeki_BM.weights.h5'
model.load_weights(weights_path)
print("Model loaded successfully.")

# 2. Load Prototype Banks
hcc_prototypes = np.load('hcc_bank_kmeans.npy')  # shape: (4, 128)
hem_prototypes = np.load('hem_bank_kmeans.npy')  # shape: (4, 128)
print(f"HCC Prototypes shape: {hcc_prototypes.shape}")


## 2. Define HCC Score GradCAM Functions

In [ ]:
def preprocess_image(img_path, target_size=(384, 384)):
    img = cv2.imread(img_path)
    if img is None:
        return None, None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, target_size)
    img_tensor = tf.convert_to_tensor(img_resized, dtype=tf.float32)
    img_tensor = tf.expand_dims(img_tensor, axis=0)
    if tf.reduce_max(img_tensor) > 1.0:
        img_tensor = img_tensor / 255.0
    return img, img_tensor

def compute_hcc_score(embedding, hcc_protos):
    # Normalize
    emb_norm = tf.math.l2_normalize(embedding, axis=-1)
    proto_norm = tf.math.l2_normalize(hcc_protos, axis=-1)
    # Cosine similarity
    sim = tf.matmul(emb_norm, proto_norm, transpose_b=True) # [B, n_protos]
    # Mean across prototypes (HCC Cosine Score)
    return tf.reduce_mean(sim, axis=-1)

def compute_hcc_gradcam(model, img_tensor, hcc_protos):
    # Split the forward pass to get gradients w.r.t patches
    # Step 1: Patch embedding
    patch_embed_layer = model.encoder.patch_embed
    tokens, fmap, gh, gw = patch_embed_layer(img_tensor, training=False)
    
    with tf.GradientTape() as tape:
        tape.watch(tokens)
        
        # Step 2: Transformer Blocks
        y = tokens
        for block in model.encoder.blocks:
            y = block(y, training=False, return_attention=False)
            
        y = model.encoder.norm(y)
        
        # In CA mode, patch_tokens remain unchanged, cls_token is updated
        patch_tokens = y[:, 1:, :]
        embedding = tf.reduce_mean(patch_tokens, axis=1) # pool_mode='gap'
        
        # Step 3: Compute HCC Score
        hcc_score = compute_hcc_score(embedding, hcc_protos)
        
    # Gradients of HCC Score w.r.t pre-attention tokens
    grads = tape.gradient(hcc_score, tokens) # [1, 145, 128]
    
    # We only care about the patches, not the CLS token slot
    patch_grads = grads[:, 1:, :] # [1, 144, 128]
    patch_tokens_pre = tokens[:, 1:, :] # [1, 144, 128]
    
    # Weight per patch (Global Average Pooling over the feature dimension of the gradient)
    weights = tf.reduce_mean(patch_grads, axis=-1) # [1, 144]
    
    # Reshape to grid
    gh_val, gw_val = int(gh), int(gw)
    heatmap = tf.reshape(weights, (gh_val, gw_val))
    
    return heatmap.numpy()

def compute_traditional_gradcam(model, img_tensor, class_index=1):
    with tf.GradientTape() as tape:
        out = model(img_tensor, training=False, return_encoder_outputs=True)
        fmap = out['feature_map']
        tape.watch(fmap)
        score = out['logits'][:, class_index]
        
    grads = tape.gradient(score, fmap)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.reduce_sum(weights * fmap, axis=-1)
    return cam[0].numpy()

def create_overlay(img, heatmap, diverging=False):
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    
    if diverging:
        # Diverging colormap (Red for positive, Blue for negative)
        vmax = np.max(np.abs(heatmap))
        if vmax > 0:
            heatmap = heatmap / vmax
        
        pos_mask = np.maximum(heatmap, 0)
        neg_mask = np.maximum(-heatmap, 0)
        
        # Base image
        overlay = img.copy().astype(np.float32) / 255.0
        
        # Red channel for positive
        overlay[:, :, 0] = np.clip(overlay[:, :, 0] + pos_mask * 0.6, 0, 1)
        # Blue channel for negative
        overlay[:, :, 2] = np.clip(overlay[:, :, 2] + neg_mask * 0.6, 0, 1)
        
        return overlay
    else:
        # Standard ReLU GradCAM
        heatmap = np.maximum(heatmap, 0)
        if np.max(heatmap) > 0:
            heatmap = heatmap / np.max(heatmap)
            
        heatmap_uint8 = np.uint8(255 * heatmap)
        jet = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
        jet = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB) / 255.0
        
        overlay = 0.5 * (img.astype(np.float32)/255.0) + 0.5 * jet
        return np.clip(overlay, 0, 1)


## 3. Visualize Example HCC Cases

In [ ]:
example_dir = '../../example_images_hcc_case'
image_files = [f for f in os.listdir(example_dir) if f.endswith(('.jpg', '.png'))]

for img_file in image_files:
    img_path = os.path.join(example_dir, img_file)
    img, img_tensor = preprocess_image(img_path, target_size=(input_size, input_size))
    
    if img is None:
        continue
        
    # Inference
    out = model(img_tensor, training=False, return_encoder_outputs=True)
    probs = out['probabilities'][0].numpy()
    embedding = tf.reduce_mean(out['patch_tokens'], axis=1)
    
    hcc_score = compute_hcc_score(embedding, hcc_prototypes)[0].numpy()
    hem_score = compute_hcc_score(embedding, hem_prototypes)[0].numpy()
    
    # Heatmaps
    hcc_heatmap = compute_hcc_gradcam(model, img_tensor, hcc_prototypes)
    trad_heatmap = compute_traditional_gradcam(model, img_tensor, class_index=1)
    
    # Overlays
    overlay_hcc_pos = create_overlay(img, hcc_heatmap, diverging=False)
    overlay_hcc_div = create_overlay(img, hcc_heatmap, diverging=True)
    overlay_trad = create_overlay(img, trad_heatmap, diverging=False)
    
    # Plotting
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    axes[0].imshow(img) # OpenCV to RGB (done in preprocess_image)
    axes[0].set_title(f"Original\nConf: {probs[1]:.4f}")
    axes[0].axis('off')
    
    axes[1].imshow(overlay_trad)
    axes[1].set_title("Standard GradCAM\n(Logits w.r.t CNN Feature Map)")
    axes[1].axis('off')
    
    axes[2].imshow(overlay_hcc_pos)
    axes[2].set_title(f"HCC Score GradCAM (ReLU)\nHCC Score: {hcc_score:.4f}\nΔscore: {hcc_score-hem_score:.4f}")
    axes[2].axis('off')
    
    axes[3].imshow(overlay_hcc_div)
    axes[3].set_title("HCC Score GradCAM (Diverging)\nRed: Increase, Blue: Decrease")
    axes[3].axis('off')
    
    plt.suptitle(f"File: {img_file}", fontsize=16)
    plt.tight_layout()
    plt.show()
